# 로더 사다리 테스트 노트북

`loader.py`(4단 폴백 사다리) + `extractor.py`(WorkbookIR)를 대화식으로 확인합니다.

커널: 리포 루트의 `.venv` 선택 (`uv sync`로 생성됨)

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():  # 노트북 cwd가 tests/인 경우
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from excel_to_skill.extractor import extract_workbook
from excel_to_skill.loader import UnsupportedFormatError, merges_from_xml

RAW = ROOT / "workingpaper_raw"
print(f"대상 파일: xlsx {len(list(RAW.glob('*.xlsx')))}개, xls {len([p for p in RAW.glob('*.xls') if p.suffix == '.xls'])}개")

대상 파일: xlsx 30개, xls 2개


## 1. 단일 파일 로드 — 감사계약 (일반 로드 경로)

지시서 §2의 골든 파일. 기대: `loader_path=openpyxl_normal`, 시트 3개, 외부 링크 10개.

In [2]:
ir = extract_workbook(RAW / "감사조서서식_1100~1300 감사계약.xlsx")
print("loader_path:", ir.loader_path)
print("외부 링크:", len(ir.external_links or []))
for s in ir.sheets:
    print(f"  시트 {s.name!r}: used range {s.dimensions}, 셀 {len(s.cells)}, 병합 {len(s.merged_ranges)}, 상태 {s.state}")

loader_path: openpyxl_normal
외부 링크: 10
  시트 '1100': used range A1:F13, 셀 57, 병합 8, 상태 visible
  시트 '1200': used range A1:H35, 셀 189, 병합 59, 상태 visible
  시트 '1300': used range A1:F16, 셀 73, 병합 17, 상태 visible


## 2. 셀 내용 훑어보기

값·수식·서식 플래그가 IR에 어떻게 담기는지 확인. `bold/border/fill`은 빈 입력 슬롯 보존(§4.4)의 재료.

In [3]:
sheet = ir.sheets[0]
print(f"시트 {sheet.name} 앞 25개 셀:\n")
print(f"{'좌표':<6} {'값':<40} {'타입':<4} {'bold':<5} {'border':<6} fill")
for (r, c), cell in sorted(sheet.cells.items())[:25]:
    val = repr(cell.value)[:38] if cell.value is not None else "·"
    print(f"{cell.coord:<6} {val:<40} {cell.data_type:<4} {str(cell.bold):<5} {str(cell.border):<6} {cell.fill or ''}")

시트 1100 앞 25개 셀:

좌표     값                                        타입   bold  border fill
A2     '1100 계약전 위험평가 (기준서 210)'                s    True  False  
A3     ·                                        n    True  False  
B3     ·                                        n    True  False  
C3     ·                                        n    True  False  
D3     ·                                        n    True  False  
E3     ·                                        n    True  False  
F3     ·                                        n    True  False  
A4     '회사명'                                    s    False False  
B4     ·                                        n    False True   #FFFFCC
C4     '작성자'                                    s    False False  
D4     ·                                        n    False True   
E4     '일자'                                     s    False False  
F4     ·                                        n    False True   
A5     '결산일 '                    

## 3. 수식 + 캐시값 (이중 로드 병합 확인)

V5 골든 엣지의 재료인 시트 간 참조 수식 6개가 보여야 함 (`1200!B4→1100!B4` 등).

In [4]:
for s in ir.sheets:
    for (r, c), cell in sorted(s.cells.items()):
        if cell.formula:
            print(f"{s.name}!{cell.coord}  =  {cell.formula}   → 캐시값: {cell.cached_value!r}")

1200!B4  =  ='1100'!B4   → 캐시값: 0
1200!D4  =  ='1100'!D4   → 캐시값: 0
1200!B5  =  ='1100'!B5   → 캐시값: datetime.time(0, 0)
1300!B4  =  ='1100'!B4   → 캐시값: 0
1300!D4  =  ='1200'!D4   → 캐시값: 0
1300!B5  =  ='1100'!B5   → 캐시값: datetime.time(0, 0)


## 4. 정의된 이름 오염 관찰 (§2 함정 4)

기대: 전역 1,363개(V6 골든) + 시트 스코프 594개. 깨진 참조·90년대 경로·이메일 샘플 확인.

In [5]:
g = [d for d in ir.defined_names if d.scope is None]
l = [d for d in ir.defined_names if d.scope is not None]
print(f"전역 {len(g)}개 (기대 1363) / 시트 스코프 {len(l)}개\n")
broken = [d for d in g if d.value and "#REF!" in d.value]
legacy = [d for d in g if d.value and ("C:\\" in d.value or ".mdb" in d.value.lower())]
print(f"#REF! 깨진 참조: {len(broken)}개, 레거시 경로: {len(legacy)}개\n")
print("오염 샘플 10건:")
for d in (broken[:5] + legacy[:5]):
    print(f"  {d.name!r} = {(d.value or '')[:70]}")

전역 1363개 (기대 1363) / 시트 스코프 594개

#REF! 깨진 참조: 429개, 레거시 경로: 32개

오염 샘플 10건:
  '__123Graph_D' = [1]IS!#REF!
  '__123Graph_F' = '[2]2600'!#REF!
  '__KEY2' = #REF!
  '_Key3' = #REF!
  '_MatInverse_In' = [1]IS!#REF!
  'AccessDatabase' = "C:\생산판매\long98\9802장판원본.mdb"
  'HTML_PathFile' = "C:\홈페이지\일위대가.htm"
  'HTML1_12' = "C:\My Documents\98년\1월\영업현황\시험.htm"
  'HTML10_12' = "C:\My Documents\98년\영업현황\일일현황-98.2.6.htm"
  'HTML11_12' = "C:\My Documents\98년\영업현황\일일현황-98.2.12.htm"


## 5. read_only 폴백 경로 (§2 함정 1·2)

일반 로드가 죽는 5종 중 하나. 기대: `openpyxl_read_only+xml_merge`, 병합은 XML 직파싱으로 취득, 숨김 행·열은 관찰 불가(`None`).

In [6]:
ir_ro = extract_workbook(RAW / "감사조서서식_1300A_독립성준수검토조서 2025.xlsx")
print("loader_path:", ir_ro.loader_path)
print("format_limitations:", ir_ro.format_limitations)
for s in ir_ro.sheets:
    print(f"  시트 {s.name!r}: 셀 {len(s.cells)}, 병합 {len(s.merged_ranges)}, hidden_rows={s.hidden_rows} (None=관찰 불가)")
print("\n병합 샘플:", ir_ro.sheets[0].merged_ranges[:8])

loader_path: openpyxl_read_only+xml_merge
format_limitations: openpyxl read_only 모드: 숨김 행·열 정보 관찰 불가
  시트 '1300A': 셀 554, 병합 68, hidden_rows=None (None=관찰 불가)

병합 샘플: ['A73:F73', 'A74:F74', 'A78:F78', 'A34:F34', 'A35:F35', 'A36:F36', 'A37:F37', 'A49:F49']


### 5b. XML 직파싱 교차 검증

일반 로드가 되는 파일에서 openpyxl 병합 결과와 XML 직파싱 결과가 일치하는지 대조.

In [7]:
xml_m = merges_from_xml(RAW / "감사조서서식_1100~1300 감사계약.xlsx")
for s in ir.sheets:
    match = sorted(s.merged_ranges) == sorted(xml_m.get(s.name, []))
    print(f"  {s.name}: openpyxl {len(s.merged_ranges)} vs XML {len(xml_m.get(s.name, []))} → {'일치 ✓' if match else '불일치 ✗'}")

  1100: openpyxl 8 vs XML 8 → 일치 ✓
  1200: openpyxl 59 vs XML 59 → 일치 ✓
  1300: openpyxl 17 vs XML 17 → 일치 ✓


## 6. .xls 경로 (§2 함정 5)

기대: `loader_path=xlrd`, 수식 원문 관찰 불가가 `format_limitations`에 명시(P6).

In [8]:
ir_xls = extract_workbook(RAW / "감사조서서식_7540 연결공시사항점검표 (KIFRS용)_2025.xls")
print("loader_path:", ir_xls.loader_path)
print("format_limitations:", ir_xls.format_limitations)
for s in ir_xls.sheets:
    print(f"  시트 {s.name!r}: {s.state}, used range {s.dimensions}, 셀 {len(s.cells)}, 병합 {len(s.merged_ranges)}, 숨김행 {len(s.hidden_rows or [])}")

loader_path: xlrd
format_limitations: xls(xlrd): 수식 원문 접근 불가 — 참조 그래프 관찰 불가(P6). 외부 링크 목록 관찰 불가.
  시트 '8400(2018)_수정 원본': hidden, used range A1:I2916, 셀 18031, 병합 1780, 숨김행 134
  시트 '개정사항': visible, used range A1:E16, 셀 76, 병합 9, 숨김행 0
  시트 '7540': visible, used range A1:J2383, 셀 20846, 병합 2733, 숨김행 0
  시트 '공시사항점검표 목차(2025)': visible, used range A1:L126, 셀 1448, 병합 11, 숨김행 0
  시트 '8400(2018)_업데이트(현 구조)': hidden, used range A1:N2520, 셀 16819, 병합 1654, 숨김행 56


## 7. 미지원 형식 거절 (docx / hwp)

In [9]:
for pattern in ("*.docx", "*.hwp"):
    f = next(RAW.glob(pattern), None)
    if f is None:
        continue
    try:
        extract_workbook(f)
        print(f"✗ {f.suffix} 거절 실패!")
    except UnsupportedFormatError as e:
        print(f"✓ 거절됨: {e}")

✓ 거절됨: 지원하지 않는 형식입니다: 감사조서서식_3900B 핵심감사사항 초안.docx (.docx). 지원 형식: xlsx, xls. 변환 후 재시도하십시오.
✓ 거절됨: 지원하지 않는 형식입니다: 감사조서서식_8300B 서면진술서 (KIFRS용)_통합감사 2025.hwp (.hwp). 지원 형식: xlsx, xls. 변환 후 재시도하십시오.


## 8. 32종 전수 스위프 (약 1~2분)

기대: 실패 0, read_only 계열 5종, xlrd 2종, 수식 합계 ≈ 4,000.

In [10]:
failures, rows = [], []
for p in sorted(RAW.glob("*.xls*")):
    try:
        w = extract_workbook(p)
    except Exception as e:
        failures.append((p.name, repr(e)))
        continue
    rows.append((p.name, w.loader_path, len(w.sheets),
                 sum(len(s.cells) for s in w.sheets),
                 sum(1 for s in w.sheets for c in s.cells.values() if c.formula),
                 sum(len(s.merged_ranges) for s in w.sheets)))

print(f"{'파일':<46} {'loader_path':<30} {'시트':>4} {'셀':>7} {'수식':>5} {'병합':>5}")
for name, lp, ns, nc, nf, nm in rows:
    print(f"{name[:44]:<46} {lp:<30} {ns:>4} {nc:>7} {nf:>5} {nm:>5}")

print(f"\n성공 {len(rows)}/32, 실패 {len(failures)}")
print(f"수식 합계: {sum(r[4] for r in rows)} (참고치 ~4,000)")
print(f"read_only 계열: {sum(1 for r in rows if 'read_only' in r[1])}종 (기대 5), xlrd: {sum(1 for r in rows if r[1] == 'xlrd')}종 (기대 2)")
for name, err in failures:
    print("  ✗", name, err)

파일                                             loader_path                      시트       셀    수식    병합
감사조서서식_01 조서파일 KIFRS.xlsx                      openpyxl_normal                   3     305     0    14
감사조서서식_02 영구조서목록.xlsx                          openpyxl_normal                   1     128     0     1
감사조서서식_03 일반조서목록 KIFRS.xlsx                    openpyxl_normal                   1     288     0     1
감사조서서식_04 감사조서철 작성 및 보존.xlsx                   openpyxl_normal                   1       4     0     4
감사조서서식_1100A_계약전 위험평가표 수임 2025.xlsx            openpyxl_normal                   1    3573   780   253
감사조서서식_1100~1300 감사계약.xlsx                     openpyxl_normal                   3     319     6    84
감사조서서식_1101A_계약전 위험평가표 계속 2025.xlsx            openpyxl_normal                   1    4552   675   211
감사조서서식_1300A_독립성준수검토조서 2025.xlsx               openpyxl_read_only+xml_merge      1     554     6    68
감사조서서식_1300B_개별 감사 참여자의 독립성 준수 확인서 2025.xlsx   openpyxl_normal           